# Tutorial: Production Training Pipeline Automation for NFL Models

Audience:
- You maintain model training and deployment for the NFL prediction service.

Prerequisites:
- Python scripting, subprocess basics, and familiarity with your current `backend/train_models.py`.

Learning goals:
- Move from manual retraining to scheduled, repeatable pipeline runs.
- Validate and promote model artifacts safely.
- Add observability and rollback-friendly run directories.


## Outline

1. Define pipeline contract and run directories
2. Build train -> validate -> promote flow
3. Add scheduling (APScheduler + external schedulers)
4. Add logging/alerts and rollback strategy
5. Exercise


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import json
import shutil

REPO_ROOT = Path.cwd().resolve()
RUN_ROOT = REPO_ROOT / 'output' / 'jupyter-notebook' / 'demo_training_runs'
ACTIVE_MODELS_DIR = REPO_ROOT / 'output' / 'jupyter-notebook' / 'demo_active_models'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
ACTIVE_MODELS_DIR.mkdir(parents=True, exist_ok=True)

RUN_ROOT, ACTIVE_MODELS_DIR


## Step 1 - Define pipeline config and run result schema

Production pipelines should write each run to an immutable run folder (`run_id`) with metadata.


In [ ]:
@dataclass
class PipelineConfig:
    min_val_accuracy: float = 0.53
    min_rows: int = 1000
    keep_last_runs: int = 30

@dataclass
class RunResult:
    run_id: str
    run_dir: Path
    metrics: dict[str, Any]
    artifacts: dict[str, Path]

cfg = PipelineConfig()
cfg


## Step 2 - Build train stage (demo version)

In production, this would call your real trainer (`python -m backend.train_models ...`).


In [ ]:
def mock_train_models(run_root: Path) -> RunResult:
    run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    run_dir = run_root / run_id
    models_dir = run_dir / 'models'
    models_dir.mkdir(parents=True, exist_ok=True)

    # Simulate model artifacts
    for name in ['home_model.joblib', 'away_model.joblib', 'preprocessor.joblib', 'win_clf_calibrated.joblib']:
        (models_dir / name).write_text('artifact placeholder', encoding='utf-8')

    metrics = {
        'val_accuracy': 0.58,
        'log_loss': 0.64,
        'rows_trained': 18234,
    }

    (run_dir / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
    return RunResult(
        run_id=run_id,
        run_dir=run_dir,
        metrics=metrics,
        artifacts={k: models_dir / k for k in ['home_model.joblib', 'away_model.joblib', 'preprocessor.joblib', 'win_clf_calibrated.joblib']},
    )


## Step 3 - Validate stage (quality gate before promotion)


In [ ]:
def validate_run(result: RunResult, config: PipelineConfig) -> tuple[bool, list[str]]:
    errors: list[str] = []

    if result.metrics.get('val_accuracy', 0.0) < config.min_val_accuracy:
        errors.append('val_accuracy below threshold')
    if result.metrics.get('rows_trained', 0) < config.min_rows:
        errors.append('training rows below threshold')

    for name, path in result.artifacts.items():
        if not path.exists():
            errors.append(f'missing artifact: {name}')

    return (len(errors) == 0, errors)


## Step 4 - Promote stage (atomic replace of active artifacts)


In [ ]:
def promote_run(result: RunResult, active_models_dir: Path) -> None:
    tmp_dir = active_models_dir.parent / f'{active_models_dir.name}__tmp'
    if tmp_dir.exists():
        shutil.rmtree(tmp_dir)
    tmp_dir.mkdir(parents=True, exist_ok=True)

    for name, src in result.artifacts.items():
        shutil.copy2(src, tmp_dir / name)

    # Swap
    backup_dir = active_models_dir.parent / f'{active_models_dir.name}__backup'
    if backup_dir.exists():
        shutil.rmtree(backup_dir)
    if active_models_dir.exists():
        active_models_dir.replace(backup_dir)
    tmp_dir.replace(active_models_dir)


## Step 5 - Execute one full run


In [ ]:
result = mock_train_models(RUN_ROOT)
ok, errors = validate_run(result, cfg)

if ok:
    promote_run(result, ACTIVE_MODELS_DIR)
    status = {'run_id': result.run_id, 'promoted': True, 'errors': []}
else:
    status = {'run_id': result.run_id, 'promoted': False, 'errors': errors}

status


## Step 6 - Scheduling patterns (no manual retraining)

Use one scheduler in production. Good options:
- APScheduler inside backend process (simple)
- Windows Task Scheduler / cron calling a pipeline script
- Airflow/Prefect for larger orchestration


In [ ]:
scheduler_example = '''
from apscheduler.schedulers.asyncio import AsyncIOScheduler

scheduler = AsyncIOScheduler()
scheduler.add_job(run_training_pipeline, trigger='cron', hour=3, minute=0)
scheduler.start()
'''

task_scheduler_example = r'''
# Windows Task Scheduler action command
python -m backend.pipeline_runner --config backend/pipeline_config.json
'''

print(scheduler_example)
print(task_scheduler_example)


## Step 7 - Production checklist

- Every run writes to `runs/<run_id>/` with metrics + artifacts.
- Validate before promotion.
- Promote atomically and keep rollback backup.
- Emit run status to logs/Slack/email.
- Keep pipeline config in version control.


## Exercises

1. Add a `max_log_loss` gate to `validate_run`.
2. Add `cleanup_old_runs(run_root, keep_last_runs)` retention policy.
3. Add failure alert payload containing run_id, stage, and error list.


In [ ]:
# Exercise scaffold
def cleanup_old_runs(run_root: Path, keep_last: int = 30) -> list[str]:
    runs = sorted([p for p in run_root.iterdir() if p.is_dir()])
    to_delete = runs[:-keep_last] if len(runs) > keep_last else []
    deleted = []
    for d in to_delete:
        shutil.rmtree(d)
        deleted.append(d.name)
    return deleted

cleanup_old_runs(RUN_ROOT, keep_last=5)
